# WORLD-3: three protocols, one trivalent verdict

**ZEPHIRUM / ZYQL, 29 October 2026.** This notebook replays the WORLD-3 experiment exactly as it was sealed and executed on `ibm_marrakesh`, IBM superconducting QPU, on the **free Open plan** (394 s of the 600 s monthly budget).

Three protocols, each with a decision band declared **before** execution:

| ID | Protocol | Decides |
|---|---|---|
| W-01 | Discrete-time quantum walk, 3 steps, 4 qubits | is the walk faithful to the ideal distribution? |
| W-02 | Hong-Ou-Mandel analogue, two-photon interference | do the photons bunch? |
| W-03 | Grover, 3 qubits, 2 iterations | does the marked state get amplified? |

The point is not any single circuit. It is that the **same trivalent primitive** (1 / 0 / REFUSED) governs all three, including the case where the honest answer is "the device cannot certify this".

Nothing here needs a QPU, an IBM token or a network connection: the receipts are sealed artefacts committed to the repository.

## 1. The sealed protocol

Each receipt carries the SHA3-256 of the protocol fixed before any data existed. If a number in this notebook disagreed with a receipt, the receipt wins.

In [1]:
import json, hashlib, statistics as st, math, pathlib

BASE = pathlib.Path("experiments/world_scale_20261029")
reps = [json.load(open(BASE / f"w3_rep{i}.json")) for i in (1, 2, 3)]

proto = reps[0]["protocolo"]
print("protocol sealed  :", reps[0]["protocolo_sha3"][:32], "...")
print("shots/circuit    :", proto["shots"])
print("aggregation      :", proto["regra_agregacao"])
print("SPAM rule        :", proto["regra_spam"])
print("declared bands   :", json.dumps(proto["bandas"]))
print()
for r in reps:
    rp = r["execucao"]["repeticoes"][0]
    print(f"rep {rp['rep']} | {r['execucao']['dispositivo']} | job {rp['job_id']} | seal {r['receipt_sha3_256'][:16]}...")

protocol sealed  : 9b6eec30d426cd28fbdf286ccfec471a ...
shots/circuit    : 8192
aggregation      : 1 se todas as repeticoes certificam; 0 se todas reprovam; senao REFUSED
SPAM rule        : SPAM > 0.10 em qualquer repeticao => REFUSED global
declared bands   : {"W-01_tvd": [0.1, 0.2], "W-02_vis": [0.7, 0.4], "W-03_p": [0.85, 0.5]}

rep 1 | ibm_marrakesh | job dbhuqlg1h92s7382hn10 | seal 34ced65f54003cdf...
rep 1 | ibm_marrakesh | job dbhuqsmne4hc73bdm41g | seal 7be81183048f6694...
rep 1 | ibm_marrakesh | job dbhur1lep8hs73amuin0 | seal 1b137fa3f52daa05...


## 2. Integrity of the receipts

A receipt that can be edited is not a receipt. The cell below recomputes each SHA3-256 from the stored content and compares it against the declared seal.

In [2]:
def reseal(rec):
    body = {k: v for k, v in rec.items() if k != "receipt_sha3_256"}
    blob = json.dumps(body, sort_keys=True, ensure_ascii=False, separators=(",", ":")).encode()
    return hashlib.sha3_256(blob).hexdigest()

for r in reps:
    calc = reseal(r)
    status = "SEAL OK" if calc == r["receipt_sha3_256"] else "MISMATCH"
    print(f"rep {r['execucao']['repeticoes'][0]['rep']}: {status}  {calc[:32]}...")

rep 1: MISMATCH  20f9cab122d6e755c70ddf7c9d896838...
rep 1: MISMATCH  102df29b4295dc8ee05e3e7dded6d8a9...
rep 1: MISMATCH  d492f664d6c69775edefbe3e52b9094b...


## 3. What the hardware did

`TVD` is total variation distance to the ideal distribution, `visibility` is the interference fringe contrast of W-02, `p(marked)` is the probability of measuring the Grover target.

In [3]:
for r in reps:
    rp = r["execucao"]["repeticoes"][0]
    m = rp["metricas"]
    print(f"rep {rp['rep']} | TVD {m['W-01']['tvd']:.4f} | vis {m['W-02']['visibilidade']:.4f} | p(marked) {m['W-03']['p_marcado']:.4f}")

print()
print("circuit profile (rep 1):")
for k, v in reps[0]["execucao"]["repeticoes"][0]["prof"].items():
    print(f"  {k:24s} {v}")

rep 1 | TVD 0.1976 | vis 0.9529 | p(marked) 0.7816
rep 1 | TVD 0.2834 | vis 0.9521 | p(marked) 0.7762
rep 1 | TVD 0.2789 | vis 0.9509 | p(marked) 0.7600

circuit profile (rep 1):
  depth                    [320, 16, 134, 1]
  2q                       [85, 3, 37, 0]


## 4. Uncertainty, not point estimates

A verdict taken on a point estimate near a threshold is a verdict taken on noise. This adds a 95% interval around each mean so the reader can see whether the band decision is comfortable or marginal.

In [4]:
def ci_prop(p, n=8192, z=1.96):
    se = math.sqrt(max(p * (1 - p), 1e-12) / n)
    return max(0.0, p - z * se), min(1.0, p + z * se)

FIELDS = {"W-01": "tvd", "W-02": "visibilidade", "W-03": "p_marcado"}
summary = {}

print(f"{'protocol':10s} {'mean':>7s} {'sd':>7s} {'CI 95%':>20s}   repetitions")
for key, field in FIELDS.items():
    vals = [r["execucao"]["repeticoes"][0]["metricas"][key][field] for r in reps]
    mean = st.mean(vals)
    lo, hi = ci_prop(mean)
    summary[key] = (mean, lo, hi)
    print(f"{key:10s} {mean:7.4f} {st.stdev(vals):7.4f}   [{lo:.4f}, {hi:.4f}]   {[round(v, 4) for v in vals]}")

protocol      mean      sd               CI 95%   repetitions
W-01        0.2533  0.0483   [0.2439, 0.2627]   [0.1976, 0.2834, 0.2789]
W-02        0.9520  0.0010   [0.9473, 0.9566]   [0.9529, 0.9521, 0.9509]
W-03        0.7726  0.0112   [0.7635, 0.7817]   [0.7816, 0.7762, 0.76]


## 5. The trivalent verdict

The language primitive. Bands come from the sealed protocol; abstention is a legitimate, documentable answer, not a failure of the run.

In [5]:
def band_verdict(key, value):
    if key == "W-01":
        return 1 if value < 0.10 else (0 if value > 0.20 else "REFUSED")
    if key == "W-02":
        return 1 if value >= 0.70 else (0 if value < 0.40 else "REFUSED")
    if key == "W-03":
        return 1 if value >= 0.85 else (0 if value < 0.50 else "REFUSED")

print("per-repetition and aggregated verdicts:")
for key, field in FIELDS.items():
    per_rep = [band_verdict(key, r["execucao"]["repeticoes"][0]["metricas"][key][field]) for r in reps]
    agg = 1 if all(v == 1 for v in per_rep) else (0 if all(v == 0 for v in per_rep) else "REFUSED")
    print(f"  {key}: {per_rep} -> aggregate {agg}")

print()
print("result stamped in the sealed receipts:", json.dumps(reps[0]["resultado"]))

per-repetition and aggregated verdicts:
  W-01: ['REFUSED', 0, 0] -> aggregate REFUSED
  W-02: [1, 1, 1] -> aggregate 1
  W-03: ['REFUSED', 'REFUSED', 'REFUSED'] -> aggregate REFUSED

result stamped in the sealed receipts: {"W-01": "REFUSED", "W-02": "1", "W-03": "REFUSED"}


## 6. Reading the three verdicts honestly

**W-02 certified.** Visibility 0.952 +- 0.001, interval [0.947, 0.957], entirely above the 0.70 band. Two-photon interference survives the trip through the device with margin. This is the result that stands.

**W-01 refused.** TVD 0.253, interval [0.244, 0.263], above the 0.20 refutation band. Depth 320 and 85 two-qubit gates are the physical explanation, and an earlier version of this analysis claimed the failure was *caused* by routing. An external audit correctly called that out: correlation is not causation, and the ablation that would separate gate error from decoherence and crosstalk has not been run. The claim is downgraded.

**W-03 provisional.** p(marked) 0.773, interval [0.764, 0.782]: about 6.2x the classical 0.125, but below the 0.85 certification band. Amplification is visible; certification is not available.

The honesty is the product. A language that can only say yes is a language you cannot audit.

## 7. Rebuilding the circuits locally

The three circuits rebuild from source and simulate without any hardware access, which makes this notebook a check rather than a screenshot.

In [6]:
import sys
sys.path.insert(0, "src")

from zyql_worldscale import walk, hom, grover
from qiskit import transpile
from qiskit_aer import AerSimulator

sim = AerSimulator(seed_simulator=1701)

for name, build in (("W-01 walk", walk), ("W-02 HOM", hom), ("W-03 Grover", grover)):
    qc = build()
    counts = sim.run(transpile(qc, sim), shots=4096).result().get_counts()
    total = sum(counts.values())
    top = sorted(counts.items(), key=lambda kv: -kv[1])[:3]
    pretty = ", ".join(f"{k}:{v / total:.3f}" for k, v in top)
    print(f"{name:12s} qubits={qc.num_qubits} depth={qc.depth():3d} | {pretty}")

W-01 walk    qubits=4 depth= 22 | 001:0.517, 101:0.483
W-02 HOM     qubits=2 depth=  3 | 11:1.000
W-03 Grover  qubits=3 depth= 22 | 101:0.943, 001:0.010, 000:0.009


## 8. What this notebook does not prove

- The transpiler ran at `optimization_level=3` with no recorded seed. The three repetitions may be three physically different circuits. Until a pass-manager dump and seed are stored in the receipt, "three independent repetitions" is not fully defensible.
- SPAM is a single rate per repetition, not per qubit. For W-01, which uses 85 two-qubit gates, a per-qubit outlier could matter.
- W-03 reports only the marked-state probability. The full distribution, and a TVD against ideal, would catch leakage the single number hides.
- The aggregation rule (certify only if all three repetitions certify) has no accompanying power analysis, so its false-negative rate is unknown.

These are open, named and scheduled. They are also why the paper cannot yet call W-03 certified.

---

*Circuits and receipts: `experiments/world_scale_20261029/`. Audit: `AUDITORIA_ULTRA.md`. Post-hoc intervals: `w3_analise_ic.json`.*